In [7]:
# Importing all important libraries
import os                       # for working with files
import random
import numpy as np              # for numerical computationss
import pandas as pd             # for working with dataframes
import seaborn as sns           # for working with maps
import torch                    # Pytorch module 
import matplotlib.pyplot as plt # for plotting informations on graph and images using tensors
import torch.nn as nn           # for creating  neural networks
from torch.utils.data import DataLoader # for dataloaders 
from PIL import Image           # for checking images
import torch.nn.functional as F # for functions for calculating loss
import torchvision.transforms as transforms   # for transforming images into tensors 
from torchvision.utils import make_grid       # for data checking
from torchvision.datasets import ImageFolder  # for working with classes and images
from torchsummary import summary              # for getting the summary of our model
import tensorflow as ts 
from  tensorflow import keras
import itertools
from sklearn.metrics import precision_score, accuracy_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.preprocessing import label_binarize
from sklearn.metrics import precision_recall_curve

%matplotlib inline

In [10]:
ls

datasets/       notebook.ipynb


In [11]:
dataset = "datasets"
train_data = "datasets/train"
valid_data = "datasets/valid"
test_data = "datasets/test"

In [12]:
diseases=os.listdir(train_data)
print(diseases)

['Strawberry___healthy', 'Grape___Black_rot', 'Potato___Early_blight', 'Blueberry___healthy', 'Corn_(maize)___healthy', 'Tomato___Target_Spot', 'Peach___healthy', 'Potato___Late_blight', 'Tomato___Late_blight', 'Tomato___Tomato_mosaic_virus', 'Pepper,_bell___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Tomato___Leaf_Mold', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Cherry_(including_sour)___Powdery_mildew', 'Apple___Cedar_apple_rust', 'Tomato___Bacterial_spot', 'Grape___healthy', 'Tomato___Early_blight', 'Corn_(maize)___Common_rust_', 'Grape___Esca_(Black_Measles)', 'Raspberry___healthy', 'Tomato___healthy', 'Cherry_(including_sour)___healthy', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Apple___Apple_scab', 'Corn_(maize)___Northern_Leaf_Blight', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Peach___Bacterial_spot', 'Pepper,_bell___Bacterial_spot', 'Tomato___Septoria_leaf_spot', 'Squash___Powdery_mildew', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Apple___B

In [13]:
print(len(diseases))

38


In [14]:
import os

# Count images in each class
for disease in diseases:
    path = os.path.join(train_data, disease)
    num_images = len(os.listdir(path))
    print(f"{disease}: {num_images}")

Strawberry___healthy: 1824
Grape___Black_rot: 1888
Potato___Early_blight: 1939
Blueberry___healthy: 1816
Corn_(maize)___healthy: 1859
Tomato___Target_Spot: 1827
Peach___healthy: 1728
Potato___Late_blight: 1939
Tomato___Late_blight: 1851
Tomato___Tomato_mosaic_virus: 1790
Pepper,_bell___healthy: 1988
Orange___Haunglongbing_(Citrus_greening): 2010
Tomato___Leaf_Mold: 1882
Grape___Leaf_blight_(Isariopsis_Leaf_Spot): 1722
Cherry_(including_sour)___Powdery_mildew: 1683
Apple___Cedar_apple_rust: 1760
Tomato___Bacterial_spot: 1702
Grape___healthy: 1692
Tomato___Early_blight: 1920
Corn_(maize)___Common_rust_: 1907
Grape___Esca_(Black_Measles): 1920
Raspberry___healthy: 1781
Tomato___healthy: 1926
Cherry_(including_sour)___healthy: 1826
Tomato___Tomato_Yellow_Leaf_Curl_Virus: 1961
Apple___Apple_scab: 2016
Corn_(maize)___Northern_Leaf_Blight: 1908
Tomato___Spider_mites Two-spotted_spider_mite: 1741
Peach___Bacterial_spot: 1838
Pepper,_bell___Bacterial_spot: 1913
Tomato___Septoria_leaf_spot: 1745

In [15]:
for _ in range(10):
    random_class = random.choice(diseases)
    class_path = os.path.join(train_data, random_class)
    random_img = random.choice(os.listdir(class_path))
    img_path = os.path.join(class_path, random_img)
    
    img = Image.open(img_path)
    print(f"{random_class}/{random_img}: {img.size}, mode={img.mode}")

Tomato___Leaf_Mold/f48a7999-9eaa-4403-a2a5-533d22092d46___Crnl_L.Mold 6854.JPG: (256, 256), mode=RGB
Grape___Black_rot/305be5c1-b018-4597-ada2-aaba5ae44eef___FAM_B.Rot 3113.JPG: (256, 256), mode=RGB
Cherry_(including_sour)___healthy/76e3298c-998d-4bce-956a-f8812315e1d6___JR_HL 3979_180deg.JPG: (256, 256), mode=RGB
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot/aa261076-42a4-45b0-bc3a-e7fab382f194___RS_GLSp 4403_new30degFlipLR.JPG: (256, 256), mode=RGB
Corn_(maize)___Northern_Leaf_Blight/2405c6b8-97e3-4223-bd26-b43f7db3e167___RS_NLB 3933_180deg.JPG: (256, 256), mode=RGB
Raspberry___healthy/d471b78b-a467-4147-a890-c864293cb113___Mary_HL 6325_270deg.JPG: (256, 256), mode=RGB
Strawberry___healthy/525ea4dd-84e2-4a43-9927-41cd69543db4___RS_HL 4457_new30degFlipLR.JPG: (256, 256), mode=RGB
Cherry_(including_sour)___healthy/1ffc094d-131a-481d-a888-0965c3da7e65___JR_HL 9883 copy.JPG: (256, 256), mode=RGB
Grape___Black_rot/b49f6eaa-9503-4f6e-9aa7-fd2bb607b865___FAM_B.Rot 3034_flipLR.JPG: (256

In [20]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [21]:
train_dataset = ImageFolder(root=train_data, transform=transform)
valid_dataset = ImageFolder(root=valid_data, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

In [22]:
print(f"Classes: {len(train_dataset.classes)}")
print(f"Train images: {len(train_dataset)}")
print(f"Valid images: {len(valid_dataset)}")

# Check one batch
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}")   # Should be [32, 3, 224, 224]
print(f"Labels shape: {labels.shape}")  # Should be [32]

Classes: 38
Train images: 70295
Valid images: 17572
Batch shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])


In [23]:
class PlantDiseaseCNN(nn.Module):
    def __init__(self, num_classes=38):
        super(PlantDiseaseCNN, self).__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)   # 3 channels → 16 filters
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)  # 16 → 32 filters
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # 32 → 64 filters
        
        # Pooling
        self.pool = nn.MaxPool2d(2, 2)
        
        # Fully connected layers
        # After 3 poolings: 224 → 112 → 56 → 28
        self.fc1 = nn.Linear(64 * 28 * 28, 512)
        self.fc2 = nn.Linear(512, num_classes)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, x):
        # Conv block 1: [32, 3, 224, 224] → [32, 16, 112, 112]
        x = self.pool(F.relu(self.conv1(x)))
        
        # Conv block 2: [32, 16, 112, 112] → [32, 32, 56, 56]
        x = self.pool(F.relu(self.conv2(x)))
        
        # Conv block 3: [32, 32, 56, 56] → [32, 64, 28, 28]
        x = self.pool(F.relu(self.conv3(x)))
        
        # Flatten: [32, 64, 28, 28] → [32, 64*28*28]
        x = x.view(x.size(0), -1)
        
        # Fully connected
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

# Create model
model = PlantDiseaseCNN(num_classes=38)

# Move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Using device: {device}")
print(model)

Using device: cpu
PlantDiseaseCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=50176, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=38, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


In [24]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()  # For classification (38 classes)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer

In [25]:
num_epochs = 10

for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    # Validation
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    val_accuracy = 100 * correct / total
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss/len(train_loader):.4f}")
    print(f"  Val Accuracy: {val_accuracy:.2f}%")

Epoch 1/10
  Train Loss: 0.9386
  Val Accuracy: 89.93%
Epoch 2/10
  Train Loss: 0.3944
  Val Accuracy: 93.42%
Epoch 3/10
  Train Loss: 0.2689
  Val Accuracy: 92.91%
Epoch 4/10
  Train Loss: 0.1981
  Val Accuracy: 93.74%
Epoch 5/10
  Train Loss: 0.1628
  Val Accuracy: 93.98%
Epoch 6/10
  Train Loss: 0.1389
  Val Accuracy: 94.87%
Epoch 7/10
  Train Loss: 0.1302
  Val Accuracy: 95.61%
Epoch 8/10
  Train Loss: 0.1131
  Val Accuracy: 95.12%
Epoch 9/10
  Train Loss: 0.0975
  Val Accuracy: 95.02%
Epoch 10/10
  Train Loss: 0.0981
  Val Accuracy: 93.87%
